---
# <div style="text-align: center"> Introduction </div>
---

Along these tutorials, we will see how <span style="color:blue">**SCOPE**</span> interacts with the different parts of the code to handle the execution of computational workflows. 

These are the topics covered in each tutorial:
1) The **System** class and its sources: the **Specie**, **Cell** and **Atom** classes  
2) The Computational workflow: **Branch**, **Workflow**, **Job**, and **Computation** classes  
3) The **State** class  
4) The **Data**, **Collection** and **VNM** classes
5) The **Input_data** class, and **scope input files**
6) Running <span style="color:blue">**SCOPE**</span> - Part 1: File Structure
7) Running <span style="color:blue">**SCOPE**</span> - Part 2: Execution 
8) Running <span style="color:blue">**SCOPE**</span> - Part 3: Detailed Actions

---
# <div style="text-align: center"> Tutorial 5: The Input</div>
---

In this tutorial, we will have a look at an input file of SCOPE. The input is how we indicate Scope to interact with a **System** and perform a task. It is used both to (1) create the **Computations** that are submitted to a computational cluster, (2) to analyze the results of those **Computations**. 

**Scope Input Files** are interpreted by Scope through the **Input_data** class. In the Data folder, these input files have the .scope extension. Importantly, they're not to be confused with the input files that Scope generates to submit Quantum Espresso (.inp) or Gaussian16 (.com) computations. 

Below, we will create the scope input file for a task that we will execute in Tutorial 7

In [1]:
import os
import scope

In [2]:
## Path of Tutorial 7's Data folder. Normally, it should be "os.path.abspath('.')+'/Data/Tutorial_7/". Change if necessary
tutorial_folder = os.path.abspath('../')+'/Data/7-Tutorial_7/'

# The Input File

In [3]:
from scope.classes_input import *

In [4]:
## Input Files can be created from a file, or from a string. 
## This is an example input, read as a string. We will soon go through the different parts
## First, notice that there are four sections, initiated with '&', and terminated with '/'.
## Sections can be empty as in the example below (see &options) or even absent. In those cases, default values will be added
## Also, notice that the values are either introduced as strings 'X', lists [], booleans, or integers/floats...
## Please respect the notation used for each type of variable:
   ## ''         for strings
   ## []         for lists
   ## True/False for booleans

## Finally, lines starting with '#' will be ignored. See environment section below for an example

test_input = """
&environment
   max_jobs         = 24
   max_procs        = 24
   requested_procs  = 1
   #queue            = 'my_preferred_one'
/

&options
/

&job_data
   branch       = 'tutorial'
   workflow     = 'all_sources' 
   hierarchy    = 1
   requisites   = []
   constrains   = ['self']

   job_setup    = 'regular'
   suffix       = 'scf'
   keyword      = 'pbe scf'
   must_be_good = True

   istate       = 'initial'
   fstate       = 'initial'

/

&qc_data
   software     = 'g16'
   jobtype      = 'scf'
   functional   = 'pbe'
   basis        = 'sto-3g'
/"""

In [5]:
# IMPORTANT. Notice that, when running the 'scope_run_input' script. The input file (-i arg) must be an actual file, not a string as in the cell above
# Thus, we save this text to an actual file, so we can call it later
from scope.read_write import save_text
file_name = ''.join([tutorial_folder,"task1.scope"])
save_text(test_input, file_name)

In [6]:
## In any case, the input is interpreted and stored as INPUT_DATA class objects:
## There are 4 input sections. Which are read separately 
local_env   = Input_data(content=test_input, section="&environment", isfile=False)
options     = Input_data(content=test_input, section="&options",     isfile=False)
job_data    = Input_data(content=test_input, section="&job_data",    isfile=False)
qc_data     = Input_data(content=test_input, section="&qc_data",     isfile=False)

INPUT_DATA.READ: Section '&options' is empty. Only defaults will be taken


### Section 1: Local Environment

In [7]:
# During the creation of the Environment-class objects (the 'global' environment), the user defines the available computational partitions (queues). 
# Typically, these are the queues that the user has access to during the project.

# Some other choices are more Job-dependent, like the requested number of processors. 
# To facilitate interaction, these options can be specified here in the input file, and modified anytime...
# without having to load and modify the 'global' environment

## These are the user choices in the input file:
local_env

Formatted input interpretation: ( self -> Instance of class Input() )
---------------------------------------------------
self.Key                 | Data Type           | Value     
---------------------------------------------------
self.max_jobs            | <class 'int'>       | 24        
self.max_procs           | <class 'int'>       | 24        
self.requested_procs     | <class 'int'>       | 1         

In [8]:
# max_jobs:
    # Maximum number of SLURM jobs running simultaneously by the user.
# max_procs: 
    # Maximum number of processors that can be used simultaneously by the user
# requested_procs:
    # Ideal number of processors to be requested in this Job's computations

In [9]:
## In the input file, we commented the variable 'queue':

## queue:
    # Optional, can be used to limit the actual queues that are accessible to this job.
    # If not specified, the 'best' queue will be selected among the ones made available in the global environment.
    # Normally, the 'best' queue is determined based on the number of available processors (free/total) for each queue

### Section 2: Options

In [10]:
## These are other options related to the submission:
options = fill_options_data(options)
print(options)

## This section was empty in the "test_input", but now defaults were added:

# want_submit:  
    # Here, the user can turn off submission (want_submit = False) of jobs. 
    # If so, input files will be created, but computations won't be submitted. 
    # This is perfect for testing the input files creation

# ignore_submitted: 
    # Normally when submitting a computation, SCOPE verifies that no computation with the same name is already running. 
    # If activated (ignore_submitted = True), this check won't be done, and multiple instances of the same job will be created.
    # This might be OK in some cases, when those job, even if they have the same name, refer to different input files.

# overwrite_inputs and overwrite_logs:
    # When true, the code can overwrite files that already exist.

Formatted input interpretation: ( self -> Instance of class Input() )
---------------------------------------------------
self.Key                 | Data Type           | Value     
---------------------------------------------------
self.want_submit         | <class 'bool'>      | True      
self.ignore_submitted    | <class 'bool'>      | False     
self.overwrite_inputs    | <class 'bool'>      | True      
self.overwrite_outputs   | <class 'bool'>      | True      



### Section 3: Job Data Section

In [11]:
## This section includes the information about the JOB, and can be divided in three parts:
# - How the JOB fits in the overall workflow 
# - JOB options
# - How is the JOB related to the States

## Basically, a Job associated with this information will be created (if it doesn't exist yet) or found (if it already exists) inside each system
job_data = fill_job_data(job_data)
print(job_data)

Formatted input interpretation: ( self -> Instance of class Input() )
---------------------------------------------------
self.Key                 | Data Type           | Value     
---------------------------------------------------
self.branch              | <class 'str'>       | tutorial  
self.workflow            | <class 'str'>       | all_sources
self.hierarchy           | <class 'int'>       | 1         
self.requisites          | <class 'list'>      | []        
self.constrains          | <class 'list'>      | ['self']  
self.job_setup           | <class 'str'>       | regular   
self.suffix              | <class 'str'>       | scf       
self.keyword             | <class 'str'>       | pbe_scf   
self.must_be_good        | <class 'bool'>      | True      
self.istate              | <class 'str'>       | initial   
self.fstate              | <class 'str'>       | initial   



In [12]:
## Here's the meaning of each variable:

# --- How the JOB fits in the overall workflow ---
# branch:       to locate the job inside the system, the BRANCH must be specified. 
# workflow:     to locate the job inside the system, the WORKFLOW must be specified. If more than one, the task will be run for each workflow
# hierarchy:    expected order of the job inside the workflow
# requisites:   list of keywords of jobs that should have finished before this one can start
# constrains:   list of keywords of jobs that should have NOT finished before this one can start

# --- JOB Options ---
# job_setup:    how are COMPUTATIONS inside the job created
# suffix:       which string must be added to the input/output/sub files to identify. For example, in ABITEM_opt_r1_HS.log, "opt" is the suffix
# keyword:      a name given to this job. 
# must_be_good: whether the job can considered completed even if it did not finish correctly. 
    # if must_be_good = True, it will try hard to do the task. For instance, if the user requested an optimization, it will not stop until reaching the minimum
    # if must_be_good = False, it will perform just one attempt

# --- How is the JOB related to the States ---
# istate:       state from which the information must be extracted 
# fstate:       state where the information will be stored


### Warning (!)

Notice that the computational workflow is created on-the-fly when running tasks; workflows are always added to the selected Branch when created.If your input has:

```
&job_data  
   branch       = 'tutorial'  
   workflow     = 'all_sources' 
```

A Workflow will be created for each source in the System, and will be added to the branch "tutorial". **Be careful with this choice !!**
Instead, it is recommended to specify workflows manually rather than using shortcuts if you're not sure what you're doing. 

### Section 4: QC data

In [13]:
## Finally, we have the information of the actual Quantum Chemistry (QC) computation.
qc_data = fill_qc_data(qc_data)
print(qc_data)

## This is the information that will be passed to the QC software to create the input 

## In principle, you can update the information here (eg. you can change the basis set) and re-submit. 
## If you do so, and everything works well, the code will compare the old and new qc_datas and if an update is detected...
## ...it will resubmit even if the same job under the old qc_data was flagged as complete.
## In other words, it should be save to update the qc_data, but I haven't tested all changes.

Formatted input interpretation: ( self -> Instance of class Input() )
---------------------------------------------------
self.Key                 | Data Type           | Value     
---------------------------------------------------
self.software            | <class 'str'>       | g16       
self.jobtype             | <class 'str'>       | scf       
self.functional          | <class 'str'>       | pbe       
self.basis               | <class 'str'>       | sto-3g    
self.loose_opt           | <class 'bool'>      | False     
self.tight_opt           | <class 'bool'>      | False     
self.is_grimme           | <class 'bool'>      | False     
self.grimme_type         | <class 'str'>       | d2        



In [14]:
## Currently, the code accepts a very limited set of qc_data options. Only Quantum Espresso and Gaussian16 packages are implemented, 
## together with a very limited set of functionals, basis sets, etc...

## The implementation of functionals, basis sets, and other G16 or QE options is relatively simple, please write us for assistance in doing so.

In [15]:
## In Quantum Espresso, we have implemented different Pseudo-Potential Libraries available in the literature. 
## These are called in this section ('pseudo' variable). 
## See below for an example of the qc_data for Quantum Espresso:

#&qc_data
#   software     = 'qe'
#   version      = '7.0'
#   jobtype      = 'vc-relax'
#   functional   = 'pbe'
#   is_hubbard   = True
#   is_grimme    = True
#   grimme_type  = 'd3bj'
#   uterm        = 2.35
#   mix_beta     = 0.7
#   elec_conv    = 1e-5
#   pseudo       = 'vanderbilt'
#/

In [16]:
job_data

Formatted input interpretation: ( self -> Instance of class Input() )
---------------------------------------------------
self.Key                 | Data Type           | Value     
---------------------------------------------------
self.branch              | <class 'str'>       | tutorial  
self.workflow            | <class 'str'>       | all_sources
self.hierarchy           | <class 'int'>       | 1         
self.requisites          | <class 'list'>      | []        
self.constrains          | <class 'list'>      | ['self']  
self.job_setup           | <class 'str'>       | regular   
self.suffix              | <class 'str'>       | scf       
self.keyword             | <class 'str'>       | pbe_scf   
self.must_be_good        | <class 'bool'>      | True      
self.istate              | <class 'str'>       | initial   
self.fstate              | <class 'str'>       | initial   